# إطار عمل مايكروسوفت إيچنت — أزور أوبن إيه آي (واجهة برمجة تطبيقات الاستجابات)

في هذا المثال البرمجي، ستستخدم **إطار عمل مايكروسوفت إيچنت (MAF)** لإنشاء وكيل بسيط مدعوم بواسطة **أزور أوبن إيه آي** باستخدام **واجهة برمجة تطبيقات الاستجابات**.

> **ملاحظة الترحيل:** كان هذا المثال يستخدم سابقًا نواة دلالية مع نماذج GitHub. تم ترحيله إلى إطار عمل مايكروسوفت إيچنت، وتم استبدال نماذج GitHub (المهملة، والتي ستتوقف في يوليو 2026) بأزور أوبن إيه آي، الذي يدعم واجهة برمجة تطبيقات الاستجابات. يستهدف `OpenAIChatClient` في MAF نقطة النهاية المستقرة `/openai/v1/` الخاصة بأزور أوبن إيه آي ويستخدم واجهة برمجة تطبيقات الاستجابات بشكل افتراضي.

الغرض من هذا المثال هو توضيح الخطوات التي سيتم تطبيقها لاحقًا في أمثلة برمجية إضافية عند تنفيذ أنماط وكيلية مختلفة.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## استيراد حزم بايثون اللازمة


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## تعريف أداة

في إطار عمل وكيل مايكروسوفت، تُعتبر **الأداة** دالة بايثون عادية مزينة بـ `@tool` يمكن للوكيل استدعاؤها. فيما يلي نعرّف أداة تُرجع وجهة عطلة عشوائية وتتجنب تكرار الوجهة السابقة.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## إنشاء الوكيل

هنا، ننشئ الوكيل المسمى `TravelAgent`.

في هذا المثال، نستخدم تعليمات بسيطة جداً. لا تتردد في تعديل هذه التعليمات لملاحظة كيف يتغير سلوك الوكيل.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## تشغيل الوكيل

الآن يمكننا تشغيل الوكيل. ننشئ `AgentSession` حتى يتذكر الوكيل المحادثة عبر الأدوار، ثم نرسل مدخلين `user_inputs`. الأول يطلب رحلة؛ والثاني يقول إن المستخدم لم يعجبه الاقتراح ويطلب آخر — يستخدم الوكيل تاريخ الجلسة بالإضافة إلى أداة `get_random_destination` للرد.

يمكنك تعديل هذه الرسائل لملاحظة كيفية تفاعل الوكيل بشكل مختلف. الردود تُرسل **تدريجيًا** رمز رمز.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
